# Microsoft Evaluation

In [ ]:
EVALUATION_MICROSOFT_PROMPT = """---Role---
You are an expert tasked with evaluating two answers to the same question based on three criteria: **Comprehensiveness** and **Diversity**.
---Goal---
You will evaluate two answers to the same question based on three criteria: **Comprehensiveness**, **Diversity**, **Empowerment** and **Directness**.

- **Comprehensiveness**: How much detail does the answer provide to cover all aspects and details of the question?
- **Diversity**: How varied and rich is the answer in providing different perspectives and insights on the question?
- **Empowerment**: How well does the answer help the reader understand and make informed judgements about the question? 
- **Directness**: How specifically and clearly does the answer address the question?

For each criterion, choose the better answer (either Answer 1 or Answer 2) and explain why. Then, select an overall winner based on these three categories.

Here is the question:
{query}

Here are the two answers:

**Answer 1:**
{answer1}

**Answer 2:**
{answer2}

Evaluate both answers using the three criteria listed above and provide detailed explanations for each criterion.

Output your evaluation in the following JSON format:

{{
    "Comprehensiveness": {{
        "Winner": "[Answer 1 or Answer 2]",
        "Explanation": "[Provide explanation here]"
    }},
    "Diversity": {{
        "Winner": "[Answer 1 or Answer 2]",
        "Explanation": "[Provide explanation here]"
    }},
    "Empowerment": {{
        "Winner": "[Answer 1 or Answer 2]",
        "Explanation": "[Provide explanation here]"
    }},
    Directness": {{
        "Winner": "[Answer 1 or Answer 2]",
        "Explanation": "[Provide explanation here]"
    }},
    "Overall Winner": {{
        "Winner": "[Answer 1 or Answer 2]",
        "Explanation": "[Summarize why this answer is the overall winner based on the three criteria]"
    }}
}}"""

## Set Model

In [ ]:
AZURE_OPENAI_ENDPOINT = ""
AZURE_OPENAI_API_KEY = ""
OPENAI_API_VERSION = "2023-08-01-preview"

In [ ]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment="gpt-4o-mini-2024-07-18",
    api_version=OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

## Building par-wise from the dataframe

In [ ]:
!pwd

In [ ]:
CONSUMER_ID = "Beatris270_Bogan287_5b3645de-a2d0-d016-0839-bab3757c4c58"
SOURCE_INTERMEDIATE_PATH = "/home/baptvit/Documents/mestrado/master-experiments/evaluations/data/silver/{consumer_id}.csv"
SOURCE_GOLD_PATH = "/home/baptvit/Documents/mestrado/master-experiments/evaluations/data/gold/{consumer_id}_microsoft_eval.csv"

In [ ]:
import pandas as pd

intermediate_consumer_id = pd.read_csv(
    SOURCE_INTERMEDIATE_PATH.format(consumer_id=CONSUMER_ID)
)

In [ ]:
intermediate_consumer_id.columns

In [ ]:
intermediate_consumer_id[
    ["experiment_id", "strategy_name", "system_promt", "input", "output"]
]

In [ ]:
from itertools import combinations

import pandas as pd


def pairwise_generation(df):
    # Ensure the DataFrame contains necessary columns
    necessary_columns = {"experiment_id", "system_prompt", "input", "output"}
    # if not necessary_columns.issubset(df.columns):
    #     raise ValueError(f"The dataframe must contain columns: {necessary_columns}")

    # Result list to keep track of comparison results
    comparison_results = []

    # Generate all unique pairs of experiment ids
    unique_experiment_ids = df["experiment_id"].unique()
    experiment_pairs = combinations(unique_experiment_ids, 2)

    return experiment_pairs


# Example usage
experiment_pairs = pairwise_generation(
    intermediate_consumer_id[["experiment_id", "system_promt", "input", "output"]]
)

In [ ]:
comparison_results = []

for exp1_id, exp2_id in experiment_pairs:
    exp1_output = intermediate_consumer_id.loc[
        intermediate_consumer_id["experiment_id"] == exp1_id, "output"
    ].values[0]
    exp2_output = intermediate_consumer_id.loc[
        intermediate_consumer_id["experiment_id"] == exp2_id, "output"
    ].values[0]

    # Append the result of this comparison to the results list
    comparison_results.append(
        {
            "consumer_id": CONSUMER_ID,
            "experiment_1": exp1_id,
            "experiment_2": exp2_id,
            "experiment_1_output": exp1_output,
            "experiment_2_output": exp2_output,
        }
    )
comparison_df = pd.DataFrame(comparison_results)

In [ ]:
comparison_df

# Interate to get the results

In [ ]:
comparison_first_line = comparison_df.iloc[1]

In [ ]:
comparison_first_line

In [ ]:
comparison_first_line["consumer_id"]

In [ ]:
prompt_eval = EVALUATION_MICROSOFT_PROMPT.format(
    query=comparison_first_line["consumer_id"],
    answer1=comparison_first_line["experiment_1_output"],
    answer2=comparison_first_line["experiment_2_output"],
)

In [ ]:
from IPython.display import Markdown

In [ ]:
Markdown(comparison_first_line["experiment_1_output"])

In [ ]:
Markdown(comparison_first_line["experiment_2_output"])

In [ ]:
response = llm.invoke(prompt_eval)

In [ ]:
from IPython.display import Markdown

Markdown(response.content)

In [ ]:
import json

try:
    output_parsed = json.loads(response.content)
    comparison_first_line["model_evaluation"] = output_parsed
    comparison_first_line["comprehensiveness"] = output_parsed["Comprehensiveness"]
    comparison_first_line["Empowerment"] = output_parsed["Empowerment"]
    comparison_first_line["Diversity"] = output_parsed["Diversity"]
    comparison_first_line["Directness"] = output_parsed["Directness"]
    comparison_first_line["Overall Winner"] = output_parsed["Overall Winner"]

except:
    output_parsed = response.content
    comparison_first_line["model_evaluation"] = output_parsed
    comparison_first_line["Comprehensiveness"] = output_parsed["Comprehensiveness"]
    comparison_first_line["Empowerment"] = output_parsed["Empowerment"]
    comparison_first_line["Diversity"] = output_parsed["Diversity"]
    comparison_first_line["Directness"] = output_parsed["Directness"]
    comparison_first_line["Overall Winner"] = output_parsed["Overall Winner"]

In [ ]:
df_evaluation_per_id = pd.DataFrame([comparison_first_line, comparison_first_line])

In [ ]:
df_evaluation_per_id.to_csv(SOURCE_GOLD_PATH.format(consumer_id=CONSUMER_ID))

# Run functions

In [1]:
from microsoft_evaluation import MicrosoftEvaluationProcessor

In [4]:
CONSUMER_ID = "Allen322_Ferry570_ad134528-56a5-35fd-c37f-466ff119c625"
processor = MicrosoftEvaluationProcessor(
    consumer_id=CONSUMER_ID,
    # azure_openai_endpoint=AZURE_OPENAI_ENDPOINT,
    # azure_openai_api_key=AZURE_OPENAI_API_KEY,
    # openai_api_version=OPENAI_API_VERSION
)

In [5]:
processor.process_evaluations_incremental()

Compare: Can you provide a breakdown of my medical bills ? with Can you provide a breakdown of my medical bills ?
Compare: LexicalSearch1HopStrategy with SimilaritySearch0HopStrategy
----------------------------------------
Compare: Can you provide a breakdown of my medical bills ? with Can you provide a breakdown of my medical bills ?
Compare: LexicalSearch1HopStrategy with LexicalSearch0HopStrategy
----------------------------------------
Error processing text: Expecting ',' delimiter: line 16 column 79 (char 1350)
Error extracting winners: string indices must be integers, not 'str'
Compare: Can you provide a breakdown of my medical bills ? with Can you provide a breakdown of my medical bills ?
Compare: LexicalSearch1HopStrategy with SimilaritySearch1HopStrategy
----------------------------------------
Compare: Can you provide a breakdown of my medical bills ? with Can you provide a breakdown of my medical bills ?
Compare: SimilaritySearch0HopStrategy with LexicalSearch0HopStrategy
-

In [5]:
results_df = processor.process_evaluations()

Compare: Can you provide a breakdown of my medical bills ? with Can you provide a breakdown of my medical bills ?
Compare: LexicalSearch1HopStrategy with SimilaritySearch0HopStrategy
----------------------------------------


KeyboardInterrupt: 

In [8]:
import pandas as pd

df_gold = pd.read_csv(
    "/home/baptvit/Documents/mestrado/master-experiments/evaluations/data/gold/Beatris270_Bogan287_5b3645de-a2d0-d016-0839-bab3757c4c58_microsoft_eval.csv"
)

In [9]:
df_gold.sort_values("experiment_1")

,consumer_id,experiment_1,expriment_1_name,experiment_2,expriment_2_name,experiment_1_output,experiment_2_output,model_evaluation,response,comprehensiveness,diversity,empowerment,directness,overall_winner
47,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q1:gemini-1.5-pro:lexical_search_0_hop,LexicalSearch0HopStrategy,Q1:gemini-1.5-pro:similarity_search_1_hop,SimilaritySearch1HopStrategy,You have two active medications:\n\n* **NDA020...,This is a summary of your medication history b...,"{'Comprehensiveness': {'Winner': 'Answer 2', '...","content='```json\n{\n ""Comprehensiveness"": ...",Answer 2,Answer 2,Answer 2,Answer 1,Answer 2
44,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q1:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Q1:gemini-1.5-pro:similarity_search_1_hop,SimilaritySearch1HopStrategy,You are currently taking two medications:\n\n1...,This is a summary of your medication history b...,"{'Comprehensiveness': {'Winner': 'Answer 2', '...","content='```json\n{\n ""Comprehensiveness"": ...",Answer 2,Answer 2,Answer 2,Answer 1,Answer 2
43,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q1:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Q1:gemini-1.5-pro:lexical_search_0_hop,LexicalSearch0HopStrategy,You are currently taking two medications:\n\n1...,You have two active medications:\n\n* **NDA020...,"{'Comprehensiveness': {'Winner': 'Answer 1', '...","content='```json\n{\n ""Comprehensiveness"": ...",Answer 1,Answer 1,Answer 1,Answer 2,Answer 1
42,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q1:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Q1:gemini-1.5-pro:similarity_search_0_hop,SimilaritySearch0HopStrategy,You are currently taking two medications:\n\n1...,You are currently taking Fexofenadine hydrochl...,"{'Comprehensiveness': {'Winner': 'Answer 1', '...","content='```json\n{\n ""Comprehensiveness"": ...",Answer 1,Answer 1,Answer 1,Answer 2,Answer 1
45,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q1:gemini-1.5-pro:similarity_search_0_hop,SimilaritySearch0HopStrategy,Q1:gemini-1.5-pro:lexical_search_0_hop,LexicalSearch0HopStrategy,You are currently taking Fexofenadine hydrochl...,You have two active medications:\n\n* **NDA020...,"{'Comprehensiveness': {'Winner': 'Answer 2', '...","content='```json\n{\n ""Comprehensiveness"": ...",Answer 2,Answer 2,Answer 2,Answer 2,Answer 2
46,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q1:gemini-1.5-pro:similarity_search_0_hop,SimilaritySearch0HopStrategy,Q1:gemini-1.5-pro:similarity_search_1_hop,SimilaritySearch1HopStrategy,You are currently taking Fexofenadine hydrochl...,This is a summary of your medication history b...,"{'Comprehensiveness': {'Winner': 'Answer 2', '...","content='```json\n{\n ""Comprehensiveness"": ...",Answer 2,Answer 2,Answer 2,Answer 2,Answer 2
29,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q2:gemini-1.5-pro:lexical_search_0_hop,LexicalSearch0HopStrategy,Q2:gemini-1.5-pro:similarity_search_1_hop,SimilaritySearch1HopStrategy,"Based on your medical records from August 30, ...",You have the following documented allergies:\n...,"{'Comprehensiveness': {'Winner': 'Answer 2', '...","content='```json\n{\n ""Comprehensiveness"": ...",Answer 2,Answer 1,Answer 2,Answer 1,Answer 2
26,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q2:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Q2:gemini-1.5-pro:similarity_search_1_hop,SimilaritySearch1HopStrategy,You have the following allergies documented in...,You have the following documented allergies:\n...,"{'Comprehensiveness': {'Winner': 'Answer 2', '...","content='```json\n{\n ""Comprehensiveness"": ...",Answer 2,Answer 1,Answer 1,Answer 2,Answer 2
25,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q2:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Q2:gemini-1.5-pro:lexical_search_0_hop,LexicalSearch0HopStrategy,You have the following allergies documented in...,"Based on your medical records from August 30, ...","{'Comprehen

In [ ]:
processor.save_results(results_df)

In [ ]:
results_df.iloc[5]

In [ ]:
Markdown(results_df.iloc[5]["experiment_1_output"])

In [ ]:
Markdown(results_df.iloc[5]["model_evaluation"])

In [ ]:
str(results_df.iloc[5]["model_evaluation"]).replace("\n", "")

# Evaluate results